# iTransformer: Внимание наоборот

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/22_itransformer.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q neuralforecast pandas numpy matplotlib seaborn

## Подготовка многомерных данных

In [ ]:
import pandas as pd
import numpy as np

# Создаём многомерные синтетические данные со связанными рядами
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')

# Общий тренд для всех рядов (имитация зависимости)
base = np.cumsum(np.random.randn(365))
series_data = []

for i in range(5):
    # Каждый ряд = общий тренд + индивидуальная сезонность + шум
    y = 100 + base * (0.5 + 0.2 * i) + 20 * np.sin(np.arange(365) / 7 * 2 * np.pi + i) + np.random.randn(365) * 5
    series_data.append(pd.DataFrame({
        'unique_id': f'series_{i}',
        'ds': dates,
        'y': y
    }))

train = pd.concat(series_data, ignore_index=True)
print(f"Всего рядов: {train['unique_id'].nunique()}")
print(train.head(10))

## iTransformer: конфигурация и обучение

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import iTransformer
from neuralforecast.losses.pytorch import MAE

# Параметры
HORIZON = 16
INPUT_SIZE = 96

# Конфигурация iTransformer
model = iTransformer(
    h=HORIZON,
    input_size=INPUT_SIZE,
    n_series=None,                    # определится автоматически
    loss=MAE(),
    max_steps=1000,
    
    # Архитектура
    hidden_size=256,                  # размерность эмбеддинга D
    n_heads=4,                        # количество голов внимания
    e_layers=3,                       # количество слоёв encoder
    d_ff=512,                         # размерность FFN
    dropout=0.1,
    
    scaler_type='standard',
    random_seed=42
)

# Обучаем
nf = NeuralForecast(
    models=[model],
    freq='D'
)
nf.fit(df=train)

# Прогнозируем
forecasts = nf.predict()
print(forecasts)

## Анализ кросс-корреляций

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def analyze_cross_correlation(df, variable_col='unique_id', value_col='y'):
    """
    Анализ кросс-корреляций между переменными.
    Высокие корреляции → iTransformer может помочь.
    """
    # Преобразуем в wide format
    wide = df.pivot(index='ds', columns=variable_col, values=value_col)
    
    # Считаем корреляционную матрицу
    corr_matrix = wide.corr()
    
    # Статистики
    # Убираем диагональ (корреляция с собой)
    mask = np.ones(corr_matrix.shape, dtype=bool)
    np.fill_diagonal(mask, False)
    off_diag = corr_matrix.values[mask]
    
    print(f"Кросс-корреляции между переменными:")
    print(f"  Средняя абсолютная: {np.mean(np.abs(off_diag)):.3f}")
    print(f"  Максимальная: {np.max(np.abs(off_diag)):.3f}")
    print(f"  Доля > 0.3: {np.mean(np.abs(off_diag) > 0.3):.1%}")
    
    # Рекомендация
    if np.mean(np.abs(off_diag)) > 0.2:
        print("\n→ Высокие кросс-корреляции. iTransformer может дать преимущество.")
    else:
        print("\n→ Низкие кросс-корреляции. Channel-independent модели могут быть лучше.")
    
    return corr_matrix

corr = analyze_cross_correlation(train)

# Визуализация
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0,
            annot=True, fmt='.2f')
ax.set_title('Корреляции между рядами')
plt.tight_layout()
plt.show()

## Сравнение с PatchTST и TSMixer

In [ ]:
from neuralforecast.models import PatchTST, TSMixer

models = [
    iTransformer(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hidden_size=256,
        n_heads=4,
        e_layers=3,
        loss=MAE(),
        max_steps=1000,
        scaler_type='standard',
        random_seed=42
    ),
    PatchTST(
        h=HORIZON,
        input_size=INPUT_SIZE,
        patch_len=16,
        stride=8,
        hidden_size=128,
        n_heads=4,
        e_layers=3,
        loss=MAE(),
        max_steps=1000,
        scaler_type='standard',
        random_seed=42
    ),
    TSMixer(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_block=4,
        ff_dim=64,
        loss=MAE(),
        max_steps=1000,
        scaler_type='standard',
        random_seed=42
    )
]

nf = NeuralForecast(models=models, freq='D')
nf.fit(df=train)
forecasts = nf.predict()

# Сравнение
print("Прогнозы:")
print(forecasts.head(20))

## Визуализация прогнозов

In [ ]:
# Выбираем один ряд для визуализации
sample_uid = 'series_0'
history = train[train['unique_id'] == sample_uid].tail(50)
forecast_data = forecasts[forecasts['unique_id'] == sample_uid]

fig, ax = plt.subplots(figsize=(12, 5))

# История
ax.plot(history['ds'], history['y'], label='История', color='blue')

# Прогнозы
ax.plot(forecast_data['ds'], forecast_data['iTransformer'], 
        label='iTransformer', linestyle='--', color='red')
ax.plot(forecast_data['ds'], forecast_data['PatchTST'], 
        label='PatchTST', linestyle='--', color='green')
ax.plot(forecast_data['ds'], forecast_data['TSMixer'], 
        label='TSMixer', linestyle='--', color='orange')

ax.set_title('Сравнение моделей на многомерных данных')
ax.set_xlabel('Дата')
ax.set_ylabel('Значение')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()